# 🎮📺 e스포츠 vs 전통 스포츠: 대중적 관심도 비교 분석

---

## 📋 프로젝트 개요

| 항목 | 내용 |
|------|------|
| **대주제** | e스포츠도 스포츠인가? |
| **소주제** | 대중적 관심도 비교 |
| **분석 목표** | e스포츠의 대중적 인기가 전통 스포츠에 필적하는지 데이터 기반 분석 |
| **주요 지표** | 시청자 수, 시청 시간, 팬덤 분포, 성장률 |

---

## 🎯 주요 분석 질문

1. **시청자 규모**: e스포츠 시청자 수가 얼마나 빠르게 성장하고 있는가?
2. **인기 게임**: 어떤 e스포츠 게임이 가장 많은 시청자를 확보하고 있는가?
3. **팬덤 분포**: e스포츠 팬덤의 지역적/언어별 분포는 어떠한가?
4. **성장 추이**: 연도별 시청 시간 증가율이 전통 스포츠를 상회하는가?

---

## 1️⃣ 라이브러리 로드 및 기본 설정

In [ ]:
# ============================================
# 라이브러리 임포트
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import platform

# 경고 메시지 숨김
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (플랫폼 자동 감지)
if platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 시각화 스타일 설정
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

# 데이터 경로 설정
DATA_PATH = 'data/'

print('✅ 라이브러리 로드 완료!')
print(f'📂 데이터 경로: {DATA_PATH}')

In [ ]:
# ============================================
# 공통 컬러 팔레트 정의
# ============================================
COLORS = {
    # 종목별
    'esports': '#9B59B6',      # 보라색
    'football': '#27AE60',     # 녹색
    'nfl': '#E74C3C',          # 빨간색
    'basketball': '#F39C12',   # 주황색
    
    # 게임별
    'League of Legends': '#C9AA71',
    'Dota 2': '#F44336',
    'CS:GO': '#DE9B35',
    'Valorant': '#FD4556',
    'Fortnite': '#9D4DBB',
    'PUBG': '#F2A900',
    'Overwatch': '#F99E1A',
    'Minecraft': '#7CB342',
    'GTA V': '#8BC34A',
    'Just Chatting': '#6441A5',
    
    # 언어별
    'English': '#3498DB',
    'Korean': '#E74C3C',
    'Spanish': '#F1C40F',
    'Portuguese': '#2ECC71',
    'Russian': '#9B59B6',
    'Japanese': '#E91E63',
    'Chinese': '#FF5722',
    'German': '#607D8B',
    'French': '#00BCD4',
    'Other': '#95A5A6'
}

print('✅ 스타일 설정 완료!')

---

## 2️⃣ 데이터 로드

### 📁 사용 데이터셋

| 데이터셋 | 출처 | 용도 |
|----------|------|------|
| Top Games on Twitch 2016-2023 | [Kaggle](https://www.kaggle.com/datasets/rankirsh/evolution-of-top-games-on-twitch) | 게임별 시청 시간/시청자 수 |
| Top Streamers on Twitch | [Kaggle](https://www.kaggle.com/datasets/aayushmishra1512/twitchdata) | 스트리머/팬덤 분석 |
| Football Data: Competitions | [Kaggle](https://www.kaggle.com/datasets/thedevastator/football-data-competitions-clubs-players-statist) | 리그 규모 참고 |

In [ ]:
# ============================================
# 실제 데이터 로드
# ============================================

# Twitch 게임별 데이터 (2016-2023)
twitch_games_raw = pd.read_csv(f'{DATA_PATH}Top games on Twitch 2016 - 2023/Twitch_game_data.csv')

# Twitch 글로벌 데이터
twitch_global_raw = pd.read_csv(f'{DATA_PATH}Top games on Twitch 2016 - 2023/Twitch_global_data.csv')

# Twitch 스트리머 데이터
twitch_streamers_raw = pd.read_csv(f'{DATA_PATH}twitchdata-update.csv')

# 축구 리그 데이터 (비교용)
football_competitions_raw = pd.read_csv(f'{DATA_PATH}dcereijo-player-scores/competitions')

print('✅ 데이터 로드 완료!')
print(f'\n📊 로드된 데이터셋:')
print(f'   - Twitch 게임별 데이터: {len(twitch_games_raw):,}행')
print(f'   - Twitch 글로벌 데이터: {len(twitch_global_raw):,}행')
print(f'   - Twitch 스트리머 데이터: {len(twitch_streamers_raw):,}행')
print(f'   - 축구 리그 데이터: {len(football_competitions_raw):,}행')

### 📌 데이터 구조 확인

로드된 실제 데이터의 구조를 확인합니다.

In [ ]:
# ============================================
# Twitch 게임별 데이터 구조 확인 및 전처리
# ============================================

print('='*60)
print('🎮 Twitch 게임별 시청 데이터')
print('='*60)
print(f'컬럼: {twitch_games_raw.columns.tolist()}')
display(twitch_games_raw.head())

# 데이터 전처리
twitch_games = twitch_games_raw.copy()

# 날짜 컬럼 생성
twitch_games['Date'] = pd.to_datetime(
    twitch_games['Year'].astype(str) + '-' + twitch_games['Month'].astype(str).str.zfill(2) + '-01'
)

# 컬럼명 정리
twitch_games = twitch_games.rename(columns={
    'Hours_watched': 'Hours_watched',
    'Hours_streamed': 'Hours_streamed',
    'Avg_viewers': 'Avg_viewers',
    'Peak_viewers': 'Peak_viewers',
    'Avg_channels': 'Avg_channels'
})

print(f'\n✅ Twitch 게임 데이터 전처리 완료!')
print(f'   - 기간: {twitch_games["Date"].min().strftime("%Y-%m")} ~ {twitch_games["Date"].max().strftime("%Y-%m")}')
print(f'   - 게임 수: {twitch_games["Game"].nunique()}개')
print(f'   - 총 레코드: {len(twitch_games):,}개')

In [ ]:
# ============================================
# Twitch 스트리머 데이터 구조 확인 및 전처리
# ============================================

print('='*60)
print('👤 Twitch 스트리머 데이터')
print('='*60)
print(f'컬럼: {twitch_streamers_raw.columns.tolist()}')
display(twitch_streamers_raw.head())

# 데이터 전처리
twitch_streamers = twitch_streamers_raw.copy()

# 컬럼명 정리
twitch_streamers = twitch_streamers.rename(columns={
    'Watch time(Minutes)': 'Watch_time_minutes',
    'Stream time(minutes)': 'Stream_time_minutes',
    'Peak viewers': 'Peak_viewers',
    'Average viewers': 'Avg_viewers',
    'Followers gained': 'Followers_gained',
    'Views gained': 'Views_gained'
})

# 파생 변수 생성
twitch_streamers['Watch_time_hours'] = twitch_streamers['Watch_time_minutes'] / 60
twitch_streamers['Stream_time_hours'] = twitch_streamers['Stream_time_minutes'] / 60
twitch_streamers['Viewer_ratio'] = twitch_streamers['Peak_viewers'] / twitch_streamers['Avg_viewers']

print(f'\n✅ Twitch 스트리머 데이터 전처리 완료!')
print(f'   - 스트리머 수: {len(twitch_streamers)}명')
print(f'   - 언어 종류: {twitch_streamers["Language"].nunique()}개')
print(f'\n📊 언어별 분포:')
print(twitch_streamers['Language'].value_counts().head(10))

In [ ]:
# ============================================
# Twitch 글로벌 데이터 및 전통 스포츠 비교 데이터
# ============================================

print('='*60)
print('🌐 Twitch 글로벌 통계 데이터')
print('='*60)
print(f'컬럼: {twitch_global_raw.columns.tolist()}')
display(twitch_global_raw.head())

# 글로벌 데이터 전처리
twitch_global = twitch_global_raw.copy()
twitch_global = twitch_global.rename(columns={'year': 'Year'})

# 주요 스포츠 이벤트 평균 시청자 수 (글로벌, 백만 명) - 비교용 참고 데이터
traditional_sports = pd.DataFrame({
    'Event': ['FIFA World Cup Final', 'Super Bowl', 'Champions League Final', 
              'NBA Finals', 'Olympics Opening', 'Cricket World Cup Final',
              'LoL Worlds Finals', 'Dota 2 TI Finals', 'CS:GO Major Finals', 'Fortnite World Cup'],
    'Category': ['Football', 'NFL', 'Football', 'Basketball', 'Olympics', 'Cricket',
                'Esports', 'Esports', 'Esports', 'Esports'],
    'Peak_Viewers_Million': [1500, 115, 450, 50, 3000, 500,
                             75, 45, 35, 23],
    'Year': [2022, 2024, 2023, 2023, 2024, 2023,
             2023, 2023, 2023, 2019],
    'Type': ['Traditional', 'Traditional', 'Traditional', 'Traditional', 'Traditional', 'Traditional',
             'Esports', 'Esports', 'Esports', 'Esports']
})

print('\n✅ 비교 데이터 준비 완료!')
print('\n📊 주요 스포츠 이벤트 시청자 (참고용):')
display(traditional_sports)

---

## 3️⃣ 데이터 탐색 (EDA)

In [ ]:
# ============================================
# Twitch 게임 데이터 탐색
# ============================================
print('='*60)
print('🎮 Twitch 게임별 시청 데이터 탐색')
print('='*60)
display(twitch_games.head(10))

print('\n📊 데이터 정보:')
print(f'   - 총 레코드 수: {len(twitch_games):,}')
print(f'   - 게임 수: {twitch_games["Game"].nunique()}')
print(f'   - 기간: {twitch_games["Year"].min()} ~ {twitch_games["Year"].max()}')

print('\n📊 시청 시간 기초 통계:')
display(twitch_games['Hours_watched'].describe())

In [ ]:
# ============================================
# Twitch 스트리머 데이터 탐색
# ============================================
print('='*60)
print('👤 Twitch 스트리머 데이터 탐색')
print('='*60)
display(twitch_streamers.head(10))

print('\n📊 언어별 스트리머 분포:')
display(twitch_streamers['Language'].value_counts())

print('\n📊 파트너 여부 분포:')
print(twitch_streamers['Partnered'].value_counts())

---

## 4️⃣ 데이터 전처리

In [ ]:
# ============================================
# 유틸리티 함수 정의
# ============================================

def format_number(value):
    """큰 숫자를 읽기 쉬운 형식으로 변환"""
    if pd.isna(value):
        return 'N/A'
    if value >= 1_000_000_000:
        return f"{value/1_000_000_000:.1f}B"
    elif value >= 1_000_000:
        return f"{value/1_000_000:.1f}M"
    elif value >= 1_000:
        return f"{value/1_000:.1f}K"
    else:
        return f"{value:.0f}"

def format_hours(hours):
    """시간을 읽기 쉬운 형식으로 변환"""
    if hours >= 1_000_000_000:
        return f"{hours/1_000_000_000:.1f}B hours"
    elif hours >= 1_000_000:
        return f"{hours/1_000_000:.1f}M hours"
    elif hours >= 1_000:
        return f"{hours/1_000:.1f}K hours"
    else:
        return f"{hours:.0f} hours"

# 테스트
print('함수 테스트:')
print(f"  format_number(1500000) = {format_number(1500000)}")
print(f"  format_hours(350000000) = {format_hours(350000000)}")

In [ ]:
# ============================================
# 연도별 집계 데이터 생성 (글로벌 데이터 활용)
# ============================================

# 연도별 총 시청 시간 (글로벌 데이터 기반)
yearly_total = twitch_global.groupby('Year').agg({
    'Hours_watched': 'sum',
    'Avg_viewers': 'mean',
    'Peak_viewers': 'max',
    'Streams': 'sum',
    'Games_streamed': 'mean'
}).reset_index()

# 성장률 계산
yearly_total['Growth_rate'] = yearly_total['Hours_watched'].pct_change() * 100

print('✅ 연도별 집계 완료! (글로벌 데이터 기반)')
display(yearly_total)

In [ ]:
# ============================================
# 게임별 집계 데이터 생성
# ============================================

# 게임별 총계
game_total = twitch_games.groupby('Game').agg({
    'Hours_watched': 'sum',
    'Avg_viewers': 'mean',
    'Peak_viewers': 'max',
    'Streamers': 'sum'
}).reset_index()

game_total = game_total.sort_values('Hours_watched', ascending=False)

print('✅ 게임별 집계 완료!')
print(f'\n📊 상위 15개 게임:')
display(game_total.head(15))

---

## 5️⃣ 시각화 및 분석

### 5.1 시청 시간 성장 추이

In [ ]:
# ============================================
# 시각화 1: 연도별 총 시청 시간 추이
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 총 시청 시간 추이 (Area Chart)
axes[0].fill_between(yearly_total['Year'], yearly_total['Hours_watched']/1e9, 
                     alpha=0.4, color=COLORS['esports'])
axes[0].plot(yearly_total['Year'], yearly_total['Hours_watched']/1e9, 
             marker='o', linewidth=2.5, markersize=10, color=COLORS['esports'])

# 값 레이블
for x, y in zip(yearly_total['Year'], yearly_total['Hours_watched']/1e9):
    axes[0].annotate(f'{y:.1f}B', (x, y), textcoords='offset points', 
                     xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')

axes[0].set_xlabel('연도', fontsize=12)
axes[0].set_ylabel('총 시청 시간 (10억 시간)', fontsize=12)
axes[0].set_title('📈 Twitch 연도별 총 시청 시간', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, yearly_total['Hours_watched'].max()/1e9 * 1.3)
axes[0].grid(True, alpha=0.3)

# 오른쪽: 연간 성장률
growth_valid = yearly_total.dropna(subset=['Growth_rate'])
colors_growth = ['#27AE60' if x > 0 else '#E74C3C' for x in growth_valid['Growth_rate']]
bars = axes[1].bar(growth_valid['Year'], growth_valid['Growth_rate'], 
                   color=colors_growth, alpha=0.8, edgecolor='white', linewidth=1.5)

# 값 레이블
for bar in bars:
    height = bar.get_height()
    axes[1].annotate(f'{height:.1f}%', 
                    xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 5 if height > 0 else -15),
                    textcoords='offset points', ha='center', fontsize=9)

axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('연도', fontsize=12)
axes[1].set_ylabel('성장률 (%)', fontsize=12)
axes[1].set_title('📊 연간 성장률', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('01_yearly_viewership_trend.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# 통계 출력
print('\n📊 시청 시간 성장 분석:')
first_year = yearly_total['Year'].min()
last_year = yearly_total['Year'].max()
print(f"   - {first_year}년: {format_hours(yearly_total[yearly_total['Year']==first_year]['Hours_watched'].values[0])}")
print(f"   - {last_year}년: {format_hours(yearly_total[yearly_total['Year']==last_year]['Hours_watched'].values[0])}")
print(f"   - 평균 성장률: {yearly_total['Growth_rate'].mean():.1f}%")
if len(growth_valid) > 0:
    max_growth_idx = growth_valid['Growth_rate'].idxmax()
    print(f"   - 최고 성장률: {growth_valid.loc[max_growth_idx, 'Growth_rate']:.1f}% ({int(growth_valid.loc[max_growth_idx, 'Year'])}년)")

### 5.2 게임별 인기도 분석

In [ ]:
# ============================================
# 시각화 2: 게임별 총 시청 시간 비교
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 상위 15개 게임
top_games_df = game_total.nlargest(15, 'Hours_watched')

# 게임별 색상 매핑
game_colors = [COLORS.get(game, '#3498DB') for game in top_games_df['Game']]

# 왼쪽: 게임별 총 시청 시간 (Horizontal Bar)
bars = axes[0].barh(top_games_df['Game'], top_games_df['Hours_watched']/1e9, 
                    color=game_colors, edgecolor='white', linewidth=1)
axes[0].set_xlabel('총 시청 시간 (10억 시간)', fontsize=12)
axes[0].set_title('🎮 게임별 총 시청 시간 Top 15', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# 값 레이블
for bar, hours in zip(bars, top_games_df['Hours_watched']/1e9):
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, 
                f'{hours:.1f}B', va='center', fontsize=9)

# 오른쪽: 시청 시간 점유율 (Pie Chart) - 상위 10개
top_10 = game_total.nlargest(10, 'Hours_watched')
others = game_total.nsmallest(len(game_total)-10, 'Hours_watched')['Hours_watched'].sum()
pie_data = pd.concat([top_10[['Game', 'Hours_watched']], 
                      pd.DataFrame({'Game': ['기타'], 'Hours_watched': [others]})], ignore_index=True)

pie_colors = [COLORS.get(g, '#95A5A6') for g in pie_data['Game']]
explode = [0.03 if i == 0 else 0 for i in range(len(pie_data))]

wedges, texts, autotexts = axes[1].pie(
    pie_data['Hours_watched'], 
    labels=pie_data['Game'],
    autopct='%1.1f%%',
    colors=pie_colors,
    explode=explode,
    shadow=True,
    startangle=90,
    textprops={'fontsize': 9}
)

for autotext in autotexts:
    autotext.set_fontsize(8)
    autotext.set_fontweight('bold')

axes[1].set_title('📊 게임별 시청 시간 점유율', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('02_game_popularity.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# 순위 출력
print('\n🏆 게임별 시청 시간 순위 Top 10:')
for i, (_, row) in enumerate(game_total.head(10).iterrows(), 1):
    print(f"   {i}. {row['Game']}: {format_hours(row['Hours_watched'])}")

In [ ]:
# ============================================
# 시각화 3: 게임별 시청 시간 추이 (Line Chart)
# ============================================
fig, ax = plt.subplots(figsize=(14, 8))

# 상위 5개 게임만 표시
top_games_list = game_total.nlargest(5, 'Hours_watched')['Game'].tolist()

for game in top_games_list:
    game_data = twitch_games[twitch_games['Game'] == game].groupby('Year')['Hours_watched'].sum()
    ax.plot(game_data.index, game_data.values/1e9, 
            marker='o', linewidth=2.5, markersize=8, 
            label=game, color=COLORS.get(game, '#3498DB'))

ax.set_xlabel('연도', fontsize=12)
ax.set_ylabel('시청 시간 (10억 시간)', fontsize=12)
ax.set_title('📈 상위 5개 게임 연도별 시청 시간 추이', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

# COVID-19 기간 표시
ax.axvspan(2020, 2021.5, alpha=0.1, color='red')
ax.annotate('COVID-19\n급증 기간', xy=(2020.7, ax.get_ylim()[1]*0.85), 
            fontsize=10, ha='center', color='red')

plt.tight_layout()
plt.savefig('03_game_trend_over_time.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### 5.3 스트리머 및 팬덤 분석

In [ ]:
# ============================================
# 시각화 4: 상위 스트리머 분석
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 상위 15 스트리머 (시청 시간 기준)
top_streamers = twitch_streamers.nlargest(15, 'Watch_time_hours')

# 언어별 색상
streamer_colors = [COLORS.get(lang, '#95A5A6') for lang in top_streamers['Language']]

# 왼쪽: 시청 시간 기준
bars = axes[0].barh(top_streamers['Channel'], top_streamers['Watch_time_hours']/1e6, 
                    color=streamer_colors, edgecolor='white', linewidth=1)
axes[0].set_xlabel('총 시청 시간 (백만 시간)', fontsize=12)
axes[0].set_title('👤 Top 15 스트리머 (시청 시간 기준)', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

for bar, hours in zip(bars, top_streamers['Watch_time_hours']/1e6):
    axes[0].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2, 
                f'{hours:.1f}M', va='center', fontsize=9)

# 오른쪽: 팔로워 기준
top_followers = twitch_streamers.nlargest(15, 'Followers')
follower_colors = [COLORS.get(lang, '#95A5A6') for lang in top_followers['Language']]

bars2 = axes[1].barh(top_followers['Channel'], top_followers['Followers']/1e6, 
                     color=follower_colors, edgecolor='white', linewidth=1)
axes[1].set_xlabel('팔로워 수 (백만 명)', fontsize=12)
axes[1].set_title('👤 Top 15 스트리머 (팔로워 기준)', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

for bar, followers in zip(bars2, top_followers['Followers']/1e6):
    axes[1].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, 
                f'{followers:.1f}M', va='center', fontsize=9)

# 범례
from matplotlib.patches import Patch
unique_langs = twitch_streamers['Language'].unique()
legend_elements = [Patch(facecolor=COLORS.get(lang, '#95A5A6'), label=lang) 
                   for lang in unique_langs[:8]]  # 상위 8개 언어만
fig.legend(handles=legend_elements, loc='lower center', ncol=4, 
           bbox_to_anchor=(0.5, -0.02), fontsize=9, title='언어')

plt.tight_layout()
plt.savefig('04_top_streamers.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ============================================
# 시각화 5: 언어별 팬덤 분포
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 언어별 집계
language_stats = twitch_streamers.groupby('Language').agg({
    'Watch_time_hours': 'sum',
    'Followers': 'sum',
    'Channel': 'count'
}).reset_index()
language_stats.columns = ['Language', 'Total_Watch_Time', 'Total_Followers', 'Streamer_Count']
language_stats = language_stats.sort_values('Total_Watch_Time', ascending=False)

# 상위 10개 언어만 표시
language_stats_top = language_stats.head(10)

# 언어별 색상
lang_colors = [COLORS.get(lang, '#95A5A6') for lang in language_stats_top['Language']]

# 왼쪽: 시청 시간 파이 차트
wedges, texts, autotexts = axes[0].pie(
    language_stats_top['Total_Watch_Time'],
    labels=language_stats_top['Language'],
    autopct='%1.1f%%',
    colors=lang_colors,
    startangle=90,
    explode=[0.02]*len(language_stats_top),
    textprops={'fontsize': 9}
)
axes[0].set_title('🌍 언어별 시청 시간 분포', fontsize=14, fontweight='bold')

# 오른쪽: 팔로워 수 파이 차트
wedges2, texts2, autotexts2 = axes[1].pie(
    language_stats_top['Total_Followers'],
    labels=language_stats_top['Language'],
    autopct='%1.1f%%',
    colors=lang_colors,
    startangle=90,
    explode=[0.02]*len(language_stats_top),
    textprops={'fontsize': 9}
)
axes[1].set_title('🌍 언어별 팔로워 분포', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('05_language_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# 통계 출력
print('\n📊 언어별 팬덤 분포 (상위 10개):')
for _, row in language_stats_top.iterrows():
    pct = row['Total_Watch_Time'] / language_stats['Total_Watch_Time'].sum() * 100
    print(f"   - {row['Language']}: {pct:.1f}% (스트리머 {row['Streamer_Count']}명)")

### 5.4 전통 스포츠와의 비교

In [ ]:
# ============================================
# 시각화 6: 주요 이벤트 시청자 수 비교
# ============================================
fig, ax = plt.subplots(figsize=(14, 8))

# 색상 매핑
event_colors = ['#27AE60' if t == 'Traditional' else '#9B59B6' 
                for t in traditional_sports['Type']]

# 정렬
traditional_sports_sorted = traditional_sports.sort_values('Peak_Viewers_Million', ascending=True)
event_colors_sorted = ['#27AE60' if t == 'Traditional' else '#9B59B6' 
                       for t in traditional_sports_sorted['Type']]

# 바 차트
bars = ax.barh(traditional_sports_sorted['Event'], 
               traditional_sports_sorted['Peak_Viewers_Million'],
               color=event_colors_sorted, edgecolor='white', linewidth=1.5)

# 값 레이블
for bar, viewers in zip(bars, traditional_sports_sorted['Peak_Viewers_Million']):
    label = f'{viewers:,.0f}M' if viewers >= 1000 else f'{viewers}M'
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2, 
            label, va='center', fontsize=10)

ax.set_xlabel('최고 시청자 수 (백만 명)', fontsize=12)
ax.set_title('🏆 주요 스포츠 이벤트 최고 시청자 수 비교', fontsize=14, fontweight='bold')
ax.set_xscale('log')  # 로그 스케일

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#27AE60', label='전통 스포츠'),
                   Patch(facecolor='#9B59B6', label='e스포츠')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('06_event_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# 비교 분석
esports_max = traditional_sports[traditional_sports['Type'] == 'Esports']['Peak_Viewers_Million'].max()
traditional_max = traditional_sports[traditional_sports['Type'] == 'Traditional']['Peak_Viewers_Million'].max()
ratio = esports_max / traditional_max * 100

print('\n📊 시청자 수 비교 분석:')
print(f"   - e스포츠 최고: {esports_max}M (LoL Worlds)")
print(f"   - 전통 스포츠 최고: {traditional_max}M (Olympics)")
print(f"   - e스포츠/전통 스포츠 비율: {ratio:.1f}%")

In [ ]:
# ============================================
# 시각화 7: 카테고리별 평균 시청자 비교
# ============================================
fig, ax = plt.subplots(figsize=(12, 6))

# 카테고리별 집계
category_stats = traditional_sports.groupby('Type')['Peak_Viewers_Million'].agg(['mean', 'max', 'min'])

x = range(len(category_stats))
width = 0.25

bars1 = ax.bar([i - width for i in x], category_stats['mean'], width, 
               label='평균', color='#3498DB', alpha=0.8)
bars2 = ax.bar([i for i in x], category_stats['max'], width, 
               label='최대', color='#27AE60', alpha=0.8)
bars3 = ax.bar([i + width for i in x], category_stats['min'], width, 
               label='최소', color='#E74C3C', alpha=0.8)

ax.set_ylabel('시청자 수 (백만 명)', fontsize=12)
ax.set_title('📊 e스포츠 vs 전통 스포츠: 시청자 통계 비교', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['e스포츠', '전통 스포츠'])
ax.legend(loc='upper right')
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# 값 레이블
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.0f}M', xy=(bar.get_x() + bar.get_width()/2, height),
                   xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('07_category_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### 5.5 월별/시간대별 패턴 분석

In [ ]:
# ============================================
# 시각화 8: 월별 시청 패턴 (Heatmap)
# ============================================

# 피벗 테이블 생성 - 게임별 월평균 시청시간
monthly_pivot = twitch_games.pivot_table(
    index='Game', 
    columns='Month', 
    values='Hours_watched', 
    aggfunc='mean'
) / 1e9  # 10억 시간 단위

# 상위 10개 게임만
top_10_games = game_total.nlargest(10, 'Hours_watched')['Game'].tolist()
monthly_pivot_top = monthly_pivot.loc[monthly_pivot.index.isin(top_10_games)]

fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(monthly_pivot_top, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '시청 시간 (10억 시간)'})

ax.set_xlabel('월', fontsize=12)
ax.set_ylabel('게임', fontsize=12)
ax.set_title('📅 게임별 월평균 시청 시간 히트맵', fontsize=14, fontweight='bold')
ax.set_xticklabels(['1월', '2월', '3월', '4월', '5월', '6월', 
                   '7월', '8월', '9월', '10월', '11월', '12월'])

plt.tight_layout()
plt.savefig('08_monthly_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# 분석
print('\n📊 월별 패턴 분석:')
monthly_total = twitch_games.groupby('Month')['Hours_watched'].mean()
peak_month = monthly_total.idxmax()
low_month = monthly_total.idxmin()
print(f"   - 최고 시청 월: {int(peak_month)}월")
print(f"   - 최저 시청 월: {int(low_month)}월")

### 5.6 종합 대시보드

In [ ]:
# ============================================
# 시각화 9: 종합 대시보드
# ============================================
fig = plt.figure(figsize=(18, 14))

# 레이아웃 설정
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# 1. 연도별 성장 추이 (상단 전체)
ax1 = fig.add_subplot(gs[0, :])
ax1.fill_between(yearly_total['Year'], yearly_total['Hours_watched']/1e9, 
                 alpha=0.4, color=COLORS['esports'])
ax1.plot(yearly_total['Year'], yearly_total['Hours_watched']/1e9, 
         marker='o', linewidth=2.5, markersize=8, color=COLORS['esports'])
ax1.set_ylabel('시청 시간 (10억 시간)')
ax1.set_title('📈 Twitch 시청 시간 성장 추이', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
for x, y in zip(yearly_total['Year'], yearly_total['Hours_watched']/1e9):
    ax1.annotate(f'{y:.1f}B', (x, y), textcoords='offset points', 
                 xytext=(0, 8), ha='center', fontsize=9)

# 2. 게임별 시청 시간 (중단 왼쪽)
ax2 = fig.add_subplot(gs[1, 0])
top5 = game_total.nlargest(5, 'Hours_watched')
colors_top5 = [COLORS.get(g, '#3498DB') for g in top5['Game']]
ax2.barh(top5['Game'], top5['Hours_watched']/1e9, color=colors_top5)
ax2.set_xlabel('시청 시간 (10억 시간)')
ax2.set_title('🎮 Top 5 게임', fontsize=12, fontweight='bold')
ax2.invert_yaxis()

# 3. 게임별 점유율 (중단 중앙)
ax3 = fig.add_subplot(gs[1, 1])
ax3.pie(top5['Hours_watched'], labels=top5['Game'], autopct='%1.1f%%',
        colors=colors_top5, startangle=90, textprops={'fontsize': 8})
ax3.set_title('📊 Top 5 점유율', fontsize=12, fontweight='bold')

# 4. 언어별 분포 (중단 오른쪽)
ax4 = fig.add_subplot(gs[1, 2])
lang_top5 = language_stats.head(5)
lang_colors_pie = [COLORS.get(l, '#95A5A6') for l in lang_top5['Language']]
ax4.pie(lang_top5['Total_Watch_Time'], labels=lang_top5['Language'],
        autopct='%1.1f%%', colors=lang_colors_pie, startangle=90, textprops={'fontsize': 9})
ax4.set_title('🌍 언어별 분포', fontsize=12, fontweight='bold')

# 5. 성장률 비교 (하단 왼쪽)
ax5 = fig.add_subplot(gs[2, 0])
growth_valid_plot = yearly_total.dropna(subset=['Growth_rate'])
growth_colors = ['#27AE60' if x > 0 else '#E74C3C' for x in growth_valid_plot['Growth_rate']]
ax5.bar(growth_valid_plot['Year'], growth_valid_plot['Growth_rate'], color=growth_colors)
ax5.axhline(y=0, color='black', linewidth=0.5)
ax5.set_ylabel('성장률 (%)')
ax5.set_title('📊 연간 성장률', fontsize=12, fontweight='bold')

# 6. 전통 스포츠 비교 (하단 중앙-오른쪽)
ax6 = fig.add_subplot(gs[2, 1:])
esports_events = traditional_sports[traditional_sports['Type'] == 'Esports'].nlargest(3, 'Peak_Viewers_Million')
trad_events = traditional_sports[traditional_sports['Type'] == 'Traditional'].nlargest(3, 'Peak_Viewers_Million')

x = range(3)
width = 0.35
ax6.bar([i - width/2 for i in x], trad_events['Peak_Viewers_Million'], width, 
        label='전통 스포츠', color='#27AE60', alpha=0.8)
ax6.bar([i + width/2 for i in x], esports_events['Peak_Viewers_Million'], width,
        label='e스포츠', color='#9B59B6', alpha=0.8)
ax6.set_ylabel('최고 시청자 (백만 명)')
ax6.set_title('🏆 주요 이벤트 시청자 비교 (Top 3)', fontsize=12, fontweight='bold')
ax6.set_xticks(x)
ax6.set_xticklabels(['1위', '2위', '3위'])
ax6.legend()
ax6.set_yscale('log')

plt.suptitle('📺 e스포츠 대중적 관심도 종합 분석 (실제 Twitch 데이터)', fontsize=18, fontweight='bold', y=1.01)
plt.savefig('09_comprehensive_dashboard.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---

## 6️⃣ 분석 결과 요약

In [ ]:
# ============================================
# 최종 비교 테이블
# ============================================

# 데이터 기반 주요 지표 계산
last_year = yearly_total['Year'].max()
first_year = yearly_total['Year'].min()
last_year_hours = yearly_total[yearly_total['Year']==last_year]['Hours_watched'].values[0]
avg_growth = yearly_total['Growth_rate'].mean()
top_game = game_total.iloc[0]['Game']
top_language = language_stats.iloc[0]['Language']

# e스포츠 주요 지표
esports_stats = {
    f'총 시청 시간 ({last_year}년)': format_hours(last_year_hours),
    '평균 연간 성장률': f"{avg_growth:.1f}%",
    '최고 성장률': f"{yearly_total['Growth_rate'].max():.1f}%",
    '인기 게임 1위': top_game,
    '주요 언어': top_language,
    '분석 스트리머 수': f"{len(twitch_streamers):,}명",
    '분석 게임 수': f"{twitch_games['Game'].nunique():,}개",
    '최대 이벤트 시청자': f"{traditional_sports[traditional_sports['Type']=='Esports']['Peak_Viewers_Million'].max()}M"
}

print('='*70)
print('📊 대중적 관심도 분석 최종 요약 (실제 데이터 기반)')
print('='*70)

for key, value in esports_stats.items():
    print(f"   {key}: {value}")

print('\n' + '='*70)
print('📈 전통 스포츠 대비 e스포츠 비교')
print('='*70)

comparison_table = pd.DataFrame({
    '지표': ['최대 이벤트 시청자', '평균 이벤트 시청자', 'Twitch 성장률', '글로벌 접근성'],
    '전통 스포츠': ['3,000M (올림픽)', '763M', '3-5%', '지역 제한적'],
    'e스포츠': ['75M (LoL Worlds)', '44.5M', f"{avg_growth:.1f}%", '글로벌 무제한'],
    '비율': ['2.5%', '5.8%', f"{avg_growth/4:.1f}배", '-']
})

display(comparison_table.set_index('지표'))

---

## 7️⃣ 결론 및 인사이트

### 📌 주요 발견사항 (실제 Twitch 데이터 기반)

#### 1. 시청자 규모
- **절대 규모**: 전통 스포츠(FIFA World Cup, 올림픽) 대비 아직 격차 존재
- **성장률**: e스포츠가 전통 스포츠보다 빠른 성장률 기록
- **COVID-19 영향**: 2020-2021년 급격한 성장 관찰

#### 2. 게임별 인기도
- **Just Chatting**: 비게임 카테고리로 높은 시청 시간 기록
- **League of Legends**: e스포츠 게임 중 꾸준한 인기 유지
- **신흥 게임**: Valorant, Fortnite 등 빠른 성장세

#### 3. 팬덤 분포
- **영어권**: 가장 큰 시장 점유
- **다양한 언어**: 스페인어, 포르투갈어, 한국어 등 글로벌 분포
- **글로벌 접근성**: 인터넷 기반으로 전세계 시청 가능

#### 4. 성장 잠재력
- **디지털 네이티브**: 젊은 세대 중심 시청
- **상호작용**: 채팅, 후원 등 실시간 소통 가능
- **플랫폼 다양화**: Twitch 외 YouTube, 아프리카TV 등

---

### 🎯 "스포츠 인정" 관점에서의 대중성 평가

| 평가 항목 | 점수 (100점) | 평가 근거 |
|-----------|-------------|----------|
| 절대 시청자 규모 | 40/100 | 전통 스포츠 대비 아직 격차 |
| 성장률 | 85/100 | 빠른 성장세 지속 |
| 글로벌 분포 | 85/100 | 전세계 고른 팬덤 분포 |
| 젊은층 인기 | 90/100 | 10-30대에서 높은 인기 |
| 미디어 노출 | 70/100 | TV/온라인 중계 증가 |
| **종합 점수** | **74/100** | 빠르게 성장 중 |

---

### 💡 분석의 한계점

1. **플랫폼 제한**: Twitch 데이터만 분석 (YouTube Gaming, 아프리카TV 미포함)
2. **전통 스포츠 비교**: 직접적인 비교 데이터 부족
3. **시청자 중복**: 동일 시청자의 중복 시청 미고려

### 💡 추가 분석 제안

1. **연령대별 시청자 분석**: 세대별 e스포츠 vs 전통 스포츠 선호도
2. **지역별 심층 분석**: 아시아, 유럽, 북미 시장 비교
3. **다중 플랫폼 분석**: YouTube, 아프리카TV 등 통합 분석

---

## 📎 부록: 추가 분석

스트리머별 효율성(시청 시간 대비 스트리밍 시간) 분석을 통해 인기 스트리머의 특성을 파악합니다.

In [ ]:
# ============================================
# 추가 분석: 스트리머 효율성 분석
# ============================================

# 스트리머 효율성 계산 (시청 시간 / 스트리밍 시간)
twitch_streamers['Efficiency'] = twitch_streamers['Watch_time_hours'] / twitch_streamers['Stream_time_hours']

# 상위 효율 스트리머
top_efficiency = twitch_streamers.nlargest(15, 'Efficiency')[['Channel', 'Language', 'Efficiency', 'Followers', 'Avg_viewers']]

print('='*60)
print('📊 스트리머 효율성 분석 (시청시간/스트리밍시간)')
print('='*60)
print('\n🏆 효율성 상위 15명 (적은 방송으로 많은 시청 확보)')
display(top_efficiency)

# 효율성 시각화
fig, ax = plt.subplots(figsize=(12, 6))

colors = [COLORS.get(lang, '#95A5A6') for lang in top_efficiency['Language']]
bars = ax.barh(top_efficiency['Channel'], top_efficiency['Efficiency'], color=colors, edgecolor='white')

ax.set_xlabel('효율성 (시청시간/스트리밍시간)', fontsize=12)
ax.set_title('👤 스트리머 효율성 Top 15', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, eff in zip(bars, top_efficiency['Efficiency']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
            f'{eff:.0f}x', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('10_streamer_efficiency.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f'\n📊 효율성 통계:')
print(f"   - 평균 효율성: {twitch_streamers['Efficiency'].mean():.1f}x")
print(f"   - 최고 효율성: {twitch_streamers['Efficiency'].max():.1f}x ({twitch_streamers.loc[twitch_streamers['Efficiency'].idxmax(), 'Channel']})")

---

## 📚 참고자료

### 사용된 데이터 파일
| 파일명 | 설명 |
|--------|------|
| `Top games on Twitch 2016 - 2023/Twitch_game_data.csv` | Twitch 게임별 시청 데이터 (2016-2023) |
| `Top games on Twitch 2016 - 2023/Twitch_global_data.csv` | Twitch 글로벌 통계 데이터 |
| `twitchdata-update.csv` | Twitch 상위 스트리머 데이터 |
| `dcereijo-player-scores/competitions` | 축구 리그 데이터 (참고용) |

### 데이터 출처
1. [Top Games on Twitch 2016-2023](https://www.kaggle.com/datasets/rankirsh/evolution-of-top-games-on-twitch)
2. [Top Streamers on Twitch](https://www.kaggle.com/datasets/aayushmishra1512/twitchdata)
3. [Football Data: Competitions](https://www.kaggle.com/datasets/thedevastator/football-data-competitions-clubs-players-statist)

### 추가 참고
- Esports Charts (https://escharts.com/)
- TwitchTracker (https://twitchtracker.com/)
- Statista - Esports Statistics

### 분석 도구
- Python 3.x
- Pandas, NumPy
- Matplotlib, Seaborn
- JupyterLab

---

**작성일**: 2025년 1월  
**프로젝트**: e스포츠도 스포츠인가? - 팀원 2 (대중적 관심도 비교)